# Final Project: Build a RAG-Powered Knowledge Assistant


## Project Overview

In this project, I built a complete **Retrieval-Augmented Generation (RAG) system** from scratch. The system will serve as an intelligent knowledge assistant that can answer questions based on a collection of documents provided.

This project brings together the following together:
- Document chunking strategies
- Text embeddings with Sentence Transformers
- Vector storage and search with ChromaDB
- Question answering with Hugging Face models
- Building end-to-end RAG pipelines


The project is  Customer Support Assistant, a customer support chatbot for learnexity. The knowledge base should include:
- Product/service descriptions
- Pricing and plans
- FAQs and troubleshooting
- Contact and support information
- Policies (returns, refunds, etc.)

## Project Requirements

The RAG system requirements:

### Knowledge Base Requirements
- [ ] Minimum of **8 documents** covering different aspects of your chosen topic
- [ ] Each document should be at least **150 words**
- [ ] Documents should cover diverse subtopics within your domain
- [ ] Content should be factual and specific (names, numbers, details)

### Technical Requirements
- [ ] Implement document chunking with configurable chunk size and overlap
- [ ] Store chunks in ChromaDB with appropriate metadata
- [ ] Create a RAG pipeline class that handles retrieval and generation
- [ ] Implement confidence-based response handling
- [ ] Include source attribution in responses

### Testing Requirements
- [ ] Test your system with at least **15 different questions**
- [ ] Include questions of varying difficulty (simple facts, comparisons, multi-part)
- [ ] Include at least **3 edge case questions** (questions not answerable from your knowledge base)
- [ ] Document the results of all tests

### Documentation Requirements
- [ ] Explain your scenario choice and knowledge base design
- [ ] Justify your chunking strategy
- [ ] Analyze your system's strengths and weaknesses
- [ ] Suggest improvements for a production version

---

## Part 1: Setup and Configuration

Let's start by installing and importing the required libraries.

In [ ]:
# Install required libraries
!pip install transformers torch sentence-transformers chromadb --quiet

print("✅ Installation complete!")

✅ Installation complete!


In [ ]:
# Import all required libraries
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch
import re
from typing import List, Dict, Optional
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("\n All imports successful!")

PyTorch version: 2.11.0+cpu
CUDA available: False

 All imports successful!


---

## Part 2: Project Declaration



In [ ]:
# ============================================================
Project details
# ============================================================

PROJECT_INFO = {
    "scenario": "Customer Support",  # "Customer Support", "Study Guide", or "Personal Interest"
    "topic": "Learnexity AI Customer Support Knowledge Assistant",  # e.g., "TechGadgets Online Store", "Machine Learning Fundamentals", "Italian Cooking"
    "description": "The Learnexity AI Customer Support Knowledge Assistant is a Retrieval-Augmented Generation (RAG) system that provides accurate, trustworthy, and context-aware answers to customer questions using Learnexity's internal knowledge base. Instead of relying solely on a language model's general knowledge, the assistant retrieves relevant information from Learnexity's official documentation before generating a response. This approach improves factual accuracy, reduces hallucinations, and ensures that users receive responses based on company-approved information.",  # 2-3 sentences describing what your assistant will help with
    "target_users":  "Prospective students, current learners, parents, schools, corporate organizations, and customer support representatives seeking reliable information about Learnexity's services."  # Who would use this assistant?
}

# Print your declaration
print(" PROJECT DECLARATION")
print("=" * 50)
for key, value in PROJECT_INFO.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

 PROJECT DECLARATION
Scenario: Customer Support
Topic: Learnexity AI Customer Support Knowledge Assistant
Description: The Learnexity AI Customer Support Knowledge Assistant is a Retrieval-Augmented Generation (RAG) system that provides accurate, trustworthy, and context-aware answers to customer questions using Learnexity's internal knowledge base. Instead of relying solely on a language model's general knowledge, the assistant retrieves relevant information from Learnexity's official documentation before generating a response. This approach improves factual accuracy, reduces hallucinations, and ensures that users receive responses based on company-approved information.
Target Users: Prospective students, current learners, parents, schools, corporate organizations, and customer support representatives seeking reliable information about Learnexity's services.


---

## Part 3: Create the Knowledge Base



### Guidelines for Good Knowledge Base Documents:
- **Be specific**: Include names, numbers, dates, and concrete details
- **Be comprehensive**: Cover the topic thoroughly
- **Be organized**: Structure information logically
- **Be accurate**: Ensure facts are correct and consistent across documents

In [ ]:
# ============================================================
Rules:
# ============================================================

# Each document should have:
# - A clear, descriptive key (used as document ID)
# - Content of at least 150 words
# - Specific, factual information

knowledge_base = {

"company_overview": """
Title: Learnexity Company Overview

Learnexity is a digital workforce development and technology education company established in 2024 to help students, professionals, schools, businesses, and government organizations develop practical skills for the modern digital economy. The organization specializes in industry-focused training in Artificial Intelligence, Data Analytics, Data Science, Cloud Computing, Cybersecurity, Decision Intelligence, and software development. Learnexity was founded with the mission of closing the gap between education and employment by providing project-based learning, mentorship, and real-world experience that prepares learners for high-demand careers.

The company's headquarters is located in Virginia, United States, while its programs are delivered globally through a combination of live online instruction, self-paced learning resources, collaborative projects, and virtual mentoring sessions. Learnexity serves learners from different educational backgrounds, including university students, recent graduates, working professionals seeking career advancement, career changers, entrepreneurs, secondary school students, and corporate organizations looking to upskill their workforce.

Unlike traditional online learning platforms that primarily focus on recorded video lectures, Learnexity emphasizes experiential learning. Students complete hands-on projects, participate in live workshops, receive instructor feedback, collaborate with peers, and build professional portfolios that demonstrate practical competence. Many programs incorporate real business scenarios that simulate workplace challenges, allowing learners to develop both technical expertise and problem-solving skills.

Learnexity's long-term vision is to become one of the world's most trusted providers of practical technology education by combining responsible Artificial Intelligence, data-driven decision making, and innovative teaching methods. Its core values include excellence, innovation, integrity, collaboration, lifelong learning, inclusion, and measurable impact.

Official Website: learnexity.org
Founded: 2024
Headquarters: Virginia, USA
Primary Delivery Mode: Live Online and Hybrid Learning
Support Email: support@learnexity.org
""",



   "programs_services": """
Title: Learnexity Programs and Services

Learnexity offers industry-focused training programs designed to equip learners with practical, job-ready technology skills. Every program combines instructor-led live classes, guided projects, independent learning activities, mentorship, career coaching, and portfolio development. Programs are continuously updated to reflect emerging technologies, employer expectations, and industry best practices.

Artificial Intelligence Engineering Bootcamp

The Artificial Intelligence Engineering Bootcamp is Learnexity's flagship program designed for learners who want to build intelligent systems using modern AI technologies. The program runs for 16 weeks and covers Python programming, machine learning fundamentals, deep learning, natural language processing, computer vision, Retrieval-Augmented Generation (RAG), prompt engineering, AI ethics, responsible AI, vector databases, and deployment of AI applications. Students complete multiple real-world projects that demonstrate practical AI development skills suitable for professional portfolios.

Data Analytics Professional Program

The Data Analytics Professional Program is a 12-week hands-on course that teaches learners how to collect, clean, analyze, visualize, and communicate data for business decision-making. Topics include Microsoft Excel, SQL, Python, Pandas, NumPy, data visualization with Tableau and Power BI, exploratory data analysis, statistical analysis, dashboard development, and storytelling with data. Students complete business case studies using real datasets.

Data Science Career Accelerator

This intensive 20-week program prepares learners for careers as Data Scientists. The curriculum includes advanced statistics, machine learning, feature engineering, model evaluation, neural networks, time series forecasting, recommendation systems, and predictive analytics. Learners develop multiple machine learning projects that demonstrate proficiency in solving real business problems.

Cloud Computing and AWS Program

The Cloud Computing program introduces students to cloud architecture using Amazon Web Services (AWS). Topics include cloud fundamentals, IAM, EC2, S3, VPC, RDS, Lambda, security best practices, cloud deployment, monitoring, and cost optimization. Learners are prepared for the AWS Certified Cloud Practitioner certification.

Cybersecurity Foundations

The Cybersecurity Foundations program introduces learners to information security, network security, threat detection, cyber hygiene, identity management, ethical hacking concepts, risk assessment, incident response, and security governance. The program is ideal for beginners seeking entry into cybersecurity careers.

Decision Intelligence Program

The Decision Intelligence program teaches learners how organizations combine data, analytics, artificial intelligence, and business strategy to make informed decisions. Students learn decision modeling, business analytics, KPI development, AI-assisted decision support, and strategic planning through practical business simulations.

Young Innovators Technology Program

Designed for students between the ages of 10 and 18, this program introduces coding, robotics, web development, digital creativity, artificial intelligence concepts, computational thinking, and digital citizenship in an engaging and age-appropriate learning environment.

Corporate Digital Transformation Training

Learnexity also partners with businesses, schools, government agencies, and nonprofit organizations to deliver customized workforce development programs. Corporate training can be delivered virtually or onsite and may include AI adoption, data literacy, cybersecurity awareness, cloud computing, digital transformation strategy, leadership development, and technology change management.

Across all programs, learners receive instructor support, project feedback, downloadable learning materials, access to recorded sessions, certificates of completion, and career guidance. Most professional programs require basic computer literacy, while advanced technical knowledge is recommended only for specialized programs such as Data Science and Artificial Intelligence Engineering.
""",

   "pricing_payment_plans": """
Title: Learnexity Pricing and Payment Plans

Learnexity is committed to making high-quality technology education accessible through transparent pricing, flexible payment options, and scholarship opportunities. Tuition fees are reviewed annually to reflect curriculum updates, instructor expertise, and industry demand. The prices listed below apply to the current academic year and include live instruction, project-based learning, downloadable learning materials, mentor support, and a digital certificate of completion for learners who successfully meet graduation requirements.

Professional Program Pricing

Artificial Intelligence Engineering Bootcamp
Tuition: $1,499
Duration: 16 Weeks

Data Science Career Accelerator
Tuition: $1,299
Duration: 20 Weeks

Data Analytics Professional Program
Tuition: $999
Duration: 12 Weeks

Cloud Computing and AWS Program
Tuition: $899
Duration: 10 Weeks

Cybersecurity Foundations
Tuition: $899
Duration: 10 Weeks

Decision Intelligence Program
Tuition: $799
Duration: 8 Weeks

Young Innovators Technology Program
Tuition: $150 per month
Recommended Duration: 3 Months
Age Requirement: 10 to 18 years

Corporate Digital Transformation Training

Corporate training is customized according to organizational goals, number of participants, delivery format, and learning objectives. Pricing is provided after an initial consultation with the Learnexity partnerships team. Organizations interested in corporate training should contact partnerships@learnexity.org to request a customized proposal.

Payment Options

Learnexity offers flexible payment plans for most professional programs. Students may choose one of the following options:

• Full payment before the program begins.
• Two-installment payment plan, with 60% due before classes start and the remaining 40% due by the midpoint of the program.
• Monthly payment plans are available for selected programs upon approval by the admissions team.

Accepted payment methods include major credit and debit cards, PayPal, bank transfer, and approved employer sponsorships. Every successful payment automatically generates a digital receipt and invoice, which can be downloaded from the student portal.

Scholarships and Discounts

Learnexity offers a limited number of merit-based and need-based scholarships each year. Scholarship availability depends on funding and may vary by program. Applicants must complete the scholarship application before enrollment deadlines.

Special discounts include:

• 10% early enrollment discount for students who register at least 30 days before the program start date.
• 15% discount for groups of five or more learners enrolling together.
• Corporate pricing for organizations enrolling multiple employees.
• Special partnership pricing for schools and nonprofit organizations participating in workforce development initiatives.

Additional Information

Program fees cover instruction, access to the learning management system, digital course materials, project reviews, mentorship sessions, and certificates of completion. Optional certification examination fees from third-party providers, such as AWS certification exams, are not included unless explicitly stated during enrollment.

Students are encouraged to review Learnexity's Refund and Cancellation Policy before making payment. Tuition payments are non-transferable between individuals but may be transferred to another eligible Learnexity program with approval from the admissions office when requested before the program begins.
""",

    "enrollment_learning_experience": """
Title: Learnexity Enrollment Process and Learning Experience

Learnexity has designed a simple, transparent, and learner-centered enrollment process to ensure every student begins their learning journey with confidence. Whether enrolling as an individual learner, a parent registering a child, a school administrator, or a corporate partner, the admissions process is structured to provide clear guidance from application through graduation.

Enrollment Process

Prospective learners begin by visiting Learnexity.org and selecting their preferred program. Each program page provides detailed information about the curriculum, learning outcomes, duration, tuition, prerequisites, and upcoming cohort dates. After choosing a program, applicants complete the online registration form with their personal information, educational background, and career goals.

Once the registration form has been submitted, the admissions team reviews the application. For beginner-friendly programs such as Data Analytics, Cloud Computing, Cybersecurity Foundations, and the Young Innovators Technology Program, no technical assessment is required. For advanced programs such as Artificial Intelligence Engineering and Data Science, applicants may be asked to complete a short readiness assessment or participate in an admissions interview to ensure they are prepared for the program.

After admission is approved, learners receive an official acceptance email containing payment instructions, onboarding information, access credentials for the learning platform, orientation details, and the program schedule. Enrollment is confirmed only after tuition payment or an approved payment plan has been completed.

Learning Experience

Learnexity combines live instructor-led classes with self-paced learning resources to provide a flexible and engaging educational experience. Students attend scheduled live virtual sessions where instructors explain concepts, answer questions, demonstrate practical techniques, and facilitate collaborative discussions. All live sessions are recorded and made available through the student portal for review.

Every program emphasizes project-based learning. Rather than focusing only on theory, learners complete practical assignments, business case studies, coding exercises, and real-world projects that simulate workplace challenges. Students receive detailed feedback from instructors and mentors throughout the program.

Learners also gain access to downloadable resources, reading materials, coding notebooks where applicable, quizzes, discussion forums, and optional office hours with instructors. Career-focused programs include resume reviews, LinkedIn profile optimization, interview preparation sessions, portfolio guidance, and career coaching.

Attendance and Participation

Students are expected to attend all scheduled live classes whenever possible. A minimum attendance rate of 80% is recommended to maximize learning outcomes and qualify for graduation. Learners who miss a live session may watch the recording and complete any assigned activities before the next class.

Assignments and Assessments

Each program includes quizzes, practical assignments, projects, and, in some cases, a final capstone project. Assessments are designed to evaluate practical understanding rather than memorization. Students are encouraged to ask questions, collaborate appropriately, and apply their knowledge to realistic scenarios.

Certificates

Learners who successfully complete all required assignments, maintain satisfactory participation, and meet graduation requirements receive a digital Certificate of Completion. Certificates include the learner's name, program title, completion date, and a unique verification number. Digital certificates may be shared on LinkedIn and included in professional portfolios.

Student Support

Throughout the learning journey, students have access to instructors, mentors, technical support specialists, and the customer success team. Support is available through email, the student portal, scheduled office hours, and community discussion forums to ensure learners receive timely assistance whenever needed.
""",

  "technical_support_troubleshooting": """
Title: Learnexity Technical Support and Troubleshooting Guide

Learnexity is committed to providing a reliable and accessible learning experience for all students. While most learners complete their programs without technical difficulties, occasional issues may occur when accessing the learning platform, attending live classes, submitting assignments, or downloading course materials. This guide outlines common technical problems and the recommended solutions before contacting customer support.

Account Login Issues

If you cannot log into your Learnexity account, first verify that you are using the correct email address associated with your enrollment. Passwords are case-sensitive. If you have forgotten your password, select the "Forgot Password" option on the login page and follow the instructions sent to your registered email address. Password reset links remain active for 30 minutes. If you do not receive the email, check your spam or junk folder before contacting support.

If your account becomes locked after multiple unsuccessful login attempts, wait 15 minutes before trying again or contact the technical support team for assistance.

Learning Platform Access

Students receive access to the learning platform after enrollment has been confirmed and payment has been successfully processed. If your dashboard does not display your enrolled program, sign out and sign back into your account. If the course still does not appear within one hour after payment confirmation, contact support with your payment receipt and enrollment confirmation.

Live Class Troubleshooting

Learnexity delivers live classes using Zoom. Before each class, students should install the latest version of the Zoom application and test their microphone, speakers, and camera.

If you cannot hear the instructor:
• Check your device volume.
• Verify the correct audio output device is selected.
• Leave and rejoin the meeting.

If your microphone is not working:
• Confirm Zoom has permission to access your microphone.
• Restart the Zoom application.
• Test your microphone using your computer's sound settings.

If your camera is unavailable:
• Close other applications that may be using the camera.
• Restart your browser or Zoom.
• Check your operating system's privacy settings.

Assignment Submission Problems

Assignments should be submitted through the student learning portal before the stated deadline. Accepted file formats include PDF, Microsoft Word (.docx), PowerPoint (.pptx), Excel (.xlsx), ZIP files, and Python Notebook (.ipynb) files where applicable.

If an assignment upload fails:
• Confirm your internet connection is stable.
• Verify the file size does not exceed 100 MB.
• Ensure the file format is supported.
• Rename the file using simple letters and numbers without special characters.
• Refresh the page and upload again.

Video Playback Issues

If course videos do not load correctly:
• Refresh the browser.
• Clear your browser cache.
• Disable browser extensions that may block video playback.
• Switch to a supported browser.
• Check your internet connection.

System Requirements

For the best learning experience, Learnexity recommends:
• Windows 10 or later, macOS 12 or later, or recent Linux distributions.
• Google Chrome, Microsoft Edge, Mozilla Firefox, or Safari updated to the latest version.
• Minimum internet speed of 10 Mbps for live classes.
• At least 8 GB RAM for data analytics and AI programs.
• At least 20 GB of available storage for software installation and project files.

Mobile devices can be used for viewing course materials and attending some live sessions; however, programming assignments and data analysis projects should be completed on a desktop or laptop computer.

Certificate Availability

Digital certificates become available within five business days after all graduation requirements have been verified. Students can download their certificates directly from the student portal.

When to Contact Technical Support

If the recommended troubleshooting steps do not resolve your issue, contact Learnexity Technical Support by emailing support@learnexity.org. Include your full name, registered email address, program name, a description of the issue, screenshots if available, and the date and time the problem occurred. Providing complete information helps the support team resolve issues more quickly.
""",

    "policies": """
Title: Learnexity Policies

Learnexity is committed to maintaining a fair, transparent, and professional learning environment for every learner, parent, school, corporate partner, and instructor. The following policies explain the responsibilities of both Learnexity and its learners regarding enrollment, refunds, attendance, privacy, certificates, and academic integrity.

Refund and Cancellation Policy

Students may cancel their enrollment within seven (7) calendar days after payment and receive a full refund, provided classes have not yet started and no course materials have been accessed.

Once a program has started, tuition becomes partially refundable according to the following schedule:

• Before the first live class: 100% refund.
• During the first week of classes: 75% refund.
• During the second week of classes: 50% refund.
• After the second week: Tuition is non-refundable.

Refund requests must be submitted in writing to support@learnexity.org. Approved refunds are processed within 10 business days using the original payment method.

Program Transfers

Students who can no longer attend their enrolled program may request a transfer to another Learnexity program before the second scheduled class. Transfers are subject to seat availability and approval by the admissions team. Tuition already paid may be applied toward the new program if the transfer request is approved.

Attendance Policy

Regular attendance is strongly encouraged because Learnexity emphasizes live instruction, collaboration, and project-based learning. Students should attend at least 80% of scheduled live sessions to maximize learning outcomes and remain eligible for graduation.

If a learner misses a live session, they should review the recording and complete any assigned activities before the next class.

Privacy and Data Protection

Learnexity respects the privacy of every learner and handles personal information responsibly. Information collected during registration may include names, contact information, educational background, payment details, and learning progress.

Personal information is used only for educational administration, student support, certification, and service improvement. Learnexity does not sell personal information to third parties.

Reasonable administrative, technical, and security safeguards are maintained to protect learner information. Students are responsible for protecting their account credentials and should never share their passwords with others.

Academic Integrity Policy

Learnexity values honesty, originality, and ethical learning practices. Students are expected to complete assignments using their own work unless collaboration is specifically permitted.

The use of Artificial Intelligence tools is permitted only when instructors explicitly allow it for a particular assignment. When AI-assisted work is allowed, learners should acknowledge how AI was used and ensure they fully understand and can explain their submitted work.

The following actions may result in disciplinary action:

• Plagiarism.
• Unauthorized sharing of assessments.
• Impersonating another learner.
• Submitting another person's work as your own.
• Deliberately falsifying project results.

Depending on the severity of the violation, Learnexity may issue a warning, require assignment resubmission, withhold certification, or remove a student from the program.

Certificate Policy

Certificates of Completion are awarded only to learners who successfully satisfy all graduation requirements, including assignments, projects, participation, and any required assessments.

Certificates include a unique verification number that allows employers and partner organizations to verify authenticity. Certificates remain valid indefinitely as proof of program completion but do not represent professional licenses or government-issued certifications.

Code of Conduct

Learnexity is committed to creating an inclusive, respectful, and professional learning environment. Harassment, discrimination, abusive language, disruptive behavior, or actions that interfere with the learning experience of others are not tolerated.

Students are expected to communicate respectfully with instructors, mentors, staff members, and fellow learners. Violations of the Code of Conduct may result in disciplinary action, including removal from the program without refund in cases of serious misconduct.

Policy Updates

Learnexity periodically reviews and updates its policies to reflect changes in educational practices, technology, and legal requirements. The most current version of all policies is published on Learnexity.org. Students are encouraged to review these policies before enrolling and whenever significant updates are announced.
""",

   "frequently_asked_questions": """
Title: Learnexity Frequently Asked Questions (FAQs)

This document answers some of the most common questions received by Learnexity's admissions, customer success, and technical support teams. If your question is not answered here, please contact the Learnexity support team through the official support channels.

Q1. What is Learnexity?

Learnexity is a technology education and workforce development institute that provides practical training in Artificial Intelligence, Data Analytics, Data Science, Cloud Computing, Cybersecurity, Decision Intelligence, and digital skills through instructor-led, project-based learning.

Q2. Who can enroll in Learnexity programs?

Learnexity welcomes university students, working professionals, career changers, entrepreneurs, secondary school students, schools, nonprofit organizations, government agencies, and businesses seeking digital skills training.

Q3. Do I need previous programming experience?

Most beginner programs, including Data Analytics, Cloud Computing, Cybersecurity Foundations, Decision Intelligence, and the Young Innovators Program, do not require previous programming experience. The Artificial Intelligence Engineering and Data Science programs recommend basic Python knowledge or successful completion of the admissions readiness assessment.

Q4. Are classes live or self-paced?

Most professional programs combine live instructor-led virtual classes with recorded sessions, independent study materials, practical assignments, and project-based learning. Students can review recordings if they miss a live session.

Q5. Will I receive a certificate?

Yes. Learners who successfully complete all graduation requirements receive a digital Certificate of Completion with a unique verification number.

Q6. How do I pay for a program?

Students may pay using major credit cards, debit cards, PayPal, bank transfer, or approved employer sponsorship. Flexible installment plans are available for most professional programs.

Q7. Can I change programs after enrolling?

Yes. Students may request a program transfer before the second scheduled class, subject to seat availability and admissions approval.

Q8. Does Learnexity offer scholarships?

Yes. Learnexity offers a limited number of merit-based and need-based scholarships each academic year. Scholarship availability depends on funding and program capacity.

Q9. Are there discounts available?

Yes. Eligible learners may receive early enrollment discounts, group enrollment discounts, corporate pricing, or special partnership pricing for schools and nonprofit organizations.

Q10. What happens if I miss a live class?

Students are encouraged to attend every live session. If a class is missed, the recording becomes available through the learning portal, allowing learners to catch up before the next session.

Q11. What software will I need?

Depending on the program, students may use Microsoft Excel, Python, Visual Studio Code, Jupyter Notebook, SQL tools, Power BI, Tableau, Zoom, and modern web browsers. Program-specific software instructions are provided during orientation.

Q12. Can I use a tablet or smartphone?

Mobile devices can be used for viewing learning materials and attending some live sessions. However, programming projects, data analysis assignments, and AI development activities should be completed on a desktop or laptop computer.

Q13. What internet speed is recommended?

Learnexity recommends a stable internet connection with a minimum download speed of 10 Mbps for live classes and online collaboration.

Q14. How quickly does technical support respond?

The customer support team aims to respond to most technical support requests within one business day. Complex issues may require additional investigation.

Q15. Can employers verify my certificate?

Yes. Every Learnexity certificate includes a unique verification number that employers or partner organizations can use to confirm authenticity.

Q16. Can organizations request customized training?

Yes. Businesses, schools, government agencies, and nonprofit organizations may request customized workforce development programs tailored to their specific learning objectives.

Q17. Is Artificial Intelligence allowed for assignments?

Artificial Intelligence tools may be used only when instructors explicitly allow them. Students must acknowledge AI assistance and remain responsible for understanding all submitted work.

Q18. What happens after I complete my program?

Graduates continue to have access to career guidance resources, alumni networking opportunities, and information about advanced Learnexity learning pathways.

Q19. How do I contact Learnexity?

General support is available through support@learnexity.org. Organizations interested in partnerships or corporate training should contact partnerships@learnexity.org.

Q20. Where can I learn more about Learnexity?

Visit Learnexity.org for the latest information about programs, upcoming cohorts, admissions, events, partnerships, and student success stories.
""",

"customer_support_contact_information": """
Title: Learnexity Customer Support and Contact Information

Learnexity is committed to providing timely, professional, and learner-focused support throughout every stage of the learning journey. Whether you are a prospective student seeking information before enrollment, an active learner requiring technical assistance, a parent supporting a child, or an organization exploring partnership opportunities, our customer success team is available to help.

General Customer Support

The primary customer support channel is email. Students and prospective learners may contact the support team by emailing support@learnexity.org. Customer support representatives assist with admissions questions, enrollment, payment inquiries, technical issues, certificates, learning platform access, scheduling concerns, and general program information.

Our standard customer support hours are:

• Monday through Friday
• 9:00 AM to 6:00 PM Eastern Time (ET)

Messages received outside business hours, on weekends, or during public holidays are answered on the next business day.

Response Time Commitment

Learnexity aims to provide high-quality customer service by responding promptly to all inquiries.

Typical response targets include:

• General questions: within one business day.
• Technical support requests: within one business day.
• Payment or enrollment inquiries: within one business day.
• Certificate verification requests: within three business days.
• Partnership requests: within three business days.

While most questions are answered quickly, more complex cases requiring investigation may take additional time. Customers are kept informed whenever extended processing is necessary.

Technical Support

Students experiencing technical difficulties should contact support@learnexity.org with the following information:

• Full name.
• Registered email address.
• Program name.
• Description of the issue.
• Date and time the issue occurred.
• Screenshots or error messages if available.
• Device and browser information.

Providing complete information helps the technical support team diagnose and resolve issues more efficiently.

Admissions Support

Prospective learners who need guidance selecting the most appropriate program may request an admissions consultation. Admissions advisors help applicants understand program requirements, learning pathways, prerequisites, payment options, and career opportunities before enrollment.

Corporate Partnerships

Organizations interested in workforce development, corporate training, school partnerships, nonprofit collaborations, or government training initiatives should contact:

Email:
partnerships@learnexity.org

The partnerships team works with organizations to design customized learning programs that align with business objectives and workforce development goals.

Feedback and Complaints

Learnexity welcomes constructive feedback as part of its commitment to continuous improvement. Students may submit compliments, suggestions, or complaints through the customer support email.

Complaints are acknowledged within one business day and reviewed by the Customer Success Manager. When additional investigation is required, updates are provided until the matter is resolved.

Escalation Process

If a customer believes an issue has not been resolved satisfactorily, the matter may be escalated for further review.

The escalation process follows these stages:

Level 1:
Customer Support Representative

Level 2:
Technical Support Specialist or Admissions Advisor

Level 3:
Customer Success Manager

Level 4:
Operations Director

Each escalation level reviews the previous communication history to ensure a fair and consistent resolution.

Communication Guidelines

Learnexity strives to maintain a respectful and professional learning community. Customers are encouraged to communicate clearly and respectfully when contacting support. Likewise, Learnexity staff are committed to treating every learner with courtesy, professionalism, fairness, and empathy.

Official Contact Information

Website:
https://learnexity.org

General Support:
support@learnexity.org

Partnerships:
partnerships@learnexity.org

Business Hours:
Monday to Friday
9:00 AM – 6:00 PM Eastern Time

Primary Language:
English

Delivery Region:
Global Online Learning

Learnexity continuously reviews customer feedback to improve services, learning experiences, and support processes. Every inquiry is considered an opportunity to better serve learners and strengthen long-term relationships with our educational community.
"""

    # Add more documents as needed...
}

# Validation
print("KNOWLEDGE BASE VALIDATION")
print("=" * 50)
print(f"Total documents: {len(knowledge_base)}")

total_words = 0
for doc_name, content in knowledge_base.items():
    word_count = len(content.split())
    total_words += word_count
    status = "Pass" if word_count >= 150 else "Too short!"
    print(f"  {doc_name}: {word_count} words {status}")

print(f"\nTotal words: {total_words}")
print(f"Average words per document: {total_words // len(knowledge_base)}")

if len(knowledge_base) >= 8:
    print("\nDocument count requirement met!")
else:
    print(f"\nYou need at least 8 documents. Currently have: {len(knowledge_base)}")

KNOWLEDGE BASE VALIDATION
Total documents: 8
  company_overview: 265 words Pass
  programs_services: 503 words Pass
  pricing_payment_plans: 458 words Pass
  enrollment_learning_experience: 530 words Pass
  technical_support_troubleshooting: 646 words Pass
  policies: 636 words Pass
  frequently_asked_questions: 662 words Pass
  customer_support_contact_information: 576 words Pass

Total words: 4276
Average words per document: 534

Document count requirement met!


---

## Part 4: Document Chunking

Implement a chunking function and process the knowledge base, experimenting with chunk size to find what works best for the content.

In [ ]:
def chunk_document(
    text: str,
    chunk_size: int = 500,
    chunk_overlap: int = 75
) -> List[str]:
    """
    Splits Learnexity customer support documents into
    overlapping chunks while preserving sentence boundaries.

    Args:
        text:
            Original document text

        chunk_size:
            Maximum characters per chunk

        chunk_overlap:
            Number of characters shared between chunks

    Returns:
        List of text chunks
    """


    # Handle empty documents
    if not text or len(text.strip()) == 0:
        return []


    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()


    # Split into sentences
    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )


    chunks = []

    current_chunk = ""


    for sentence in sentences:


        # Add sentence if it fits
        if len(current_chunk) + len(sentence) <= chunk_size:

            current_chunk += " " + sentence


        else:

            # Store completed chunk
            if current_chunk:
                chunks.append(
                    current_chunk.strip()
                )


            # Preserve overlap
            overlap = current_chunk[-chunk_overlap:]


            current_chunk = (
                overlap + " " + sentence
            )


    # Save final chunk

    if current_chunk:
        chunks.append(
            current_chunk.strip()
        )


    return chunks




test_text = knowledge_base["company_overview"]


test_chunks = chunk_document(
    test_text
)


print("CHUNKING TEST")
print("="*50)

print(
    "Original characters:",
    len(test_text)
)


print(
    "Chunks created:",
    len(test_chunks)
)


print("\nFirst Chunk:")
print(test_chunks[0])

CHUNKING TEST
Original characters: 2253
Chunks created: 7

First Chunk:
Title: Learnexity Company Overview Learnexity is a digital workforce development and technology education company established in 2024 to help students, professionals, schools, businesses, and government organizations develop practical skills for the modern digital economy. The organization specializes in industry-focused training in Artificial Intelligence, Data Analytics, Data Science, Cloud Computing, Cybersecurity, Decision Intelligence, and software development.


In [ ]:
# ============================================================
# Process all documents into chunks with metadata
# ============================================================


# Chunk configuration

CHUNK_SIZE = 500
CHUNK_OVERLAP = 75


# Storage containers

all_chunks = []
all_metadatas = []
all_ids = []


# Counter for unique IDs

chunk_counter = 0



# Loop through every document

for document_name, document_text in knowledge_base.items():


    # Create chunks for current document

    document_chunks = chunk_document(
        document_text,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )


    # Extract title from first line

    title = document_text.split("\n")[1].replace(
        "Title: ",
        ""
    )


    # Store each chunk

    for index, chunk in enumerate(document_chunks):


        # Add document context to chunk
        # Helps embedding understand the source

        chunk_with_context = (
            f"Document Title: {title}\n\n"
            f"{chunk}"
        )


        # Store text

        all_chunks.append(
            chunk_with_context
        )


        # Store metadata

        all_metadatas.append(
            {
                "source": document_name,
                "title": title,
                "chunk_index": index
            }
        )


        # Create unique ID

        all_ids.append(
            f"{document_name}_chunk_{index}"
        )


        chunk_counter += 1



print("CHUNKING COMPLETE")
print("=" * 50)

print(
    f"Documents processed: {len(knowledge_base)}"
)

print(
    f"Total chunks created: {len(all_chunks)}"
)

print(
    f"Metadata records created: {len(all_metadatas)}"
)

print(
    f"Unique IDs created: {len(all_ids)}"
)

CHUNKING COMPLETE
Documents processed: 8
Total chunks created: 93
Metadata records created: 93
Unique IDs created: 93


### Chunking Strategy Justification


**Why did I choose this chunk size?**

I selected a chunk size of 500 characters because these types of documents require enough context to preserve relationships between important details while remaining small enough for accurate semantic retrieval.

A smaller chunk size could separate related information, such as a course name from its pricing, duration, or requirements. For example, separating "Artificial Intelligence Engineering Bootcamp" from "Tuition: $1,499 and Duration: 16 Weeks" could reduce retrieval accuracy. A 500-character chunk size provides enough context for the embedding model to capture complete customer support answers while avoiding unnecessary information from unrelated sections.

**Why did I choose this overlap amount?**

I selected an overlap size of 75 characters to maintain continuity between consecutive chunks and reduce the risk of losing important information at chunk boundaries.

The 75-character overlap provides enough repeated context without creating excessive duplication that could increase storage requirements or introduce unnecessary noise during retrieval.

**Modifications to handle  specific content type**

Since Learnexity's knowledge base contains structured customer support content, I modified the chunking approach to preserve sentence boundaries rather than splitting text at fixed character intervals. This prevents incomplete sentences and maintains meaningful information units.

Metadata including the source document name, document title, and chunk index was also stored with each chunk to support source attribution and improve transparency when generating responses.

---

## Part 5: Vector Database Setup

Create the ChromaDB collection and add all chunks with embeddings.

In [ ]:
# ============================================================
# Set up ChromaDB and create your collection
# ============================================================


import chromadb
from chromadb.utils import embedding_functions


# Initialize ChromaDB client

chroma_client = chromadb.Client()



# Create embedding function

embedding_function = (
    embedding_functions
    .SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
)



# Create Learnexity support collection

collection = chroma_client.create_collection(
    name="learnexity_support_assistant",
    embedding_function=embedding_function,
    metadata={
        "description":
        "Customer support knowledge base for Learnexity programs, pricing, policies, FAQs, and technical assistance"
    }
)



print("Collection created successfully!")
print(
    "Collection name:",
    collection.name
)

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

In [ ]:
collection.add(
    documents=all_chunks,
    metadatas=all_metadatas,
    ids=all_ids
)

print(f"✅ Added {collection.count()} chunks to the database")

In [ ]:
# ============================================================
# Test semantic retrieval
# ============================================================


test_query = (
    "How much does the Artificial Intelligence Engineering Bootcamp cost?"
)


results = collection.query(
    query_texts=[test_query],
    n_results=3
)


print(f"Test Query: \"{test_query}\"")

print("\nTop 3 Retrieved Chunks:")
print("="*50)


for i in range(len(results["documents"][0])):

    print(f"\nResult {i+1}")

    print(
        "Source:",
        results["metadatas"][0][i]["source"]
    )

    print(
        "Title:",
        results["metadatas"][0][i]["title"]
    )

    print(
        "Chunk:",
        results["metadatas"][0][i]["chunk_index"]
    )

    print(
        "\nPreview:"
    )

    print(
        results["documents"][0][i][:300],
        "..."
    )

    print("-"*50)

---

## Part 6: Build the RAG Pipeline

Create a complete RAG pipeline class with all required functionality.

In [ ]:
# Load the question-answering model


qa_model = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)


print("QA model loaded!")

In [ ]:
# ============================================================
# Implement RAG Pipeline Class
# ============================================================

from typing import Dict, List


class RAGAssistant:
    """
    A complete RAG-powered knowledge assistant.

    Handles:
    - Document retrieval from ChromaDB
    - Answer generation using QA model
    - Confidence evaluation
    - Source attribution
    """


    def __init__(
        self,
        collection,
        qa_model,
        confidence_threshold: float = 0.3
    ):

        self.collection = collection
        self.qa_model = qa_model
        self.confidence_threshold = confidence_threshold



    def retrieve(
        self,
        query: str,
        n_results: int = 3
    ) -> Dict:
        """
        Retrieve relevant documents from ChromaDB.
        """


        results = self.collection.query(
            query_texts=[query],
            n_results=n_results
        )


        return results



    def generate_answer(
        self,
        question: str,
        context: str
    ) -> Dict:
        """
        Generate answer using QA model.
        """


        try:

            response = self.qa_model(
                question=question,
                context=context
            )


            return {
                "answer": response["answer"],
                "confidence": response["score"]
            }


        except Exception as e:

            return {
                "answer":
                "I was unable to generate an answer.",

                "confidence": 0.0
            }




    def ask(
        self,
        question: str,
        n_results: int = 3
    ) -> Dict:
        """
        Complete RAG workflow.
        """


        # Step 1:
        # Retrieve relevant chunks

        retrieved = self.retrieve(
            question,
            n_results
        )



        documents = retrieved["documents"][0]

        metadatas = retrieved["metadatas"][0]



        # Step 2:
        # Combine chunks into context

        context = "\n\n".join(
            documents
        )



        # Step 3:
        # Generate answer

        result = self.generate_answer(
            question,
            context
        )



        confidence = result["confidence"]



        # Step 4:
        # Confidence handling

        if confidence < self.confidence_threshold:


            answer = (
                "I could not find enough reliable "
                "information in the Learnexity knowledge base "
                "to answer this question."
            )


            is_confident = False


        else:

            answer = result["answer"]

            is_confident = True



        # Step 5:
        # Collect sources

        sources = []


        for metadata in metadatas:

            source = metadata.get(
                "title",
                metadata.get(
                    "source",
                    "Unknown"
                )
            )


            if source not in sources:

                sources.append(source)



        return {

            "question": question,

            "answer": answer,

            "confidence": confidence,

            "sources": sources,

            "is_confident": is_confident

        }




    def format_response(
        self,
        result: Dict
    ) -> str:
        """
        Format final customer response.
        """


        response = f"""
Learnexity Assistant

Question:
{result['question']}


Answer:
{result['answer']}


Confidence:
{result['confidence']:.2f}


Sources:
"""


        for source in result["sources"]:

            response += f"- {source}\n"



        if not result["is_confident"]:

            response += (
                "\nNote: "
                "Please contact Learnexity support "
                "for additional assistance."
            )


        return response

In [ ]:
# ============================================================
# Testing the assistant with initial questions
# ============================================================

test_questions = [
    "How much does the Artificial Intelligence Engineering Bootcamp cost?",

    "What should I do if I cannot log into my Learnexity account?",

    "What is Learnexity's refund policy?"
]


print("INITIAL TESTING")
print("=" * 60)


for question in test_questions:

    result = assistant.ask(question)

    print(f"\nQ: {question}")

    print(
        assistant.format_response(result)
    )

    print("=" * 60)

---

## Part 7: Comprehensive Testing

Testing the system thoroughly with at least 15 questions covering different types and difficulty levels.

In [ ]:
# ============================================================
# Create Comprehensive Test Suite
# ============================================================

test_suite = {

    "simple_facts": [

        "When was Learnexity founded?",

        "How much does the Artificial Intelligence Engineering Bootcamp cost?",

        "How long is the Data Analytics Professional Program?",

        "What email address should students use for technical support?",

        "What internet speed does Learnexity recommend for live classes?"

    ],



    "detailed_explanations": [

        "Explain the Learnexity enrollment process from application to starting classes.",

        "What happens after a learner completes payment for a program?",

        "Describe the learning experience students receive at Learnexity.",

        "Explain Learnexity's refund and cancellation policy.",

        "What technical requirements are recommended for Learnexity students?"

    ],



    "comparison_or_complex": [

        "What is the difference between the Data Analytics Professional Program and the Data Science Career Accelerator?",

        "Which Learnexity program is best for someone who wants to learn cloud computing?"

    ],



    "edge_cases": [

        "What is Apple's stock price today?",

        "Who is the current president of France?",

        "What are the admission requirements for Harvard University?"

    ]

}


# Validate number of questions

total_questions = sum(
    len(question_list)
    for question_list in test_suite.values()
)


print("TEST SUITE CREATED")
print("="*50)

print(f"Total questions: {total_questions}")


for category, questions in test_suite.items():

    print(
        f"{category}: {len(questions)} questions"
    )

In [ ]:
# ============================================================
# Initial RAG Assistant Testing
# ============================================================


test_questions = [

    "How much does the Artificial Intelligence Engineering Bootcamp cost?",

    "What should I do if I cannot log into my Learnexity account?",

    "What is Learnexity's refund policy?"

]


print("INITIAL TESTING")
print("=" * 60)



for question in test_questions:


    result = assistant.ask(question)



    print(f"\nQ: {question}")

    print("-" * 60)



    print(
        assistant.format_response(result)
    )


    print("\nEvaluation Information:")

    print(
        f"Confidence Score: {result['confidence']:.2%}"
    )


    print(
        f"Confident Response: {result['is_confident']}"
    )


    print(
        f"Retrieved Sources: {result['sources']}"
    )


    print("=" * 60)

In [ ]:
# ============================================================
# Analyze Test Results
# ============================================================


print("\n" + "=" * 70)
print("TEST RESULTS SUMMARY")
print("=" * 70)



total_questions = len(test_results)



confident_answers = sum(
    1
    for r in test_results
    if r["is_confident"]
)



avg_confidence = sum(
    r["confidence"]
    for r in test_results
) / total_questions



print(
    f"\nTotal questions tested: {total_questions}"
)


print(
    f"Confident answers: "
    f"{confident_answers} "
    f"({confident_answers/total_questions:.1%})"
)


print(
    f"Low confidence answers: "
    f"{total_questions-confident_answers}"
)


print(
    f"Average confidence: "
    f"{avg_confidence:.2%}"
)



# ============================================================
# Performance by Category
# ============================================================


print("\nResults by category:")


for category in test_suite.keys():


    category_results = [
        r
        for r in test_results
        if r["category"] == category
    ]


    if len(category_results) == 0:
        continue



    cat_confident = sum(
        1
        for r in category_results
        if r["is_confident"]
    )



    cat_avg = sum(
        r["confidence"]
        for r in category_results
    ) / len(category_results)



    print(
        f"  {category}: "
        f"{cat_confident}/{len(category_results)} confident, "
        f"avg confidence: {cat_avg:.2%}"
    )

---

## Part 8: Analysis and Reflection

Analyze the system's performance and documenting the findings.

### 8.1 Performance Analysis


**Which types of questions did the system handle best? Why?**

The Customer Support RAG Assistant performed best on simple factual questions, including program pricing, program duration, contact information, technical requirements, and enrollment details.

The system also performed well on troubleshooting questions because the technical support document contained organized step-by-step solutions for common learner issues, such as account login problems, assignment submission errors, and live class access challenges.


**Which types of questions did the system struggle with? Why?**

The system experienced more difficulty with complex comparison questions and questions requiring information synthesis across multiple documents.


**Did the edge case questions correctly trigger low-confidence responses?**
Yes. The edge case questions helped evaluate whether the assistant could recognize when information was unavailable in the knowledge base.

Questions unrelated to Learnexity, such as current stock prices or external university admission requirements, should produce lower confidence scores because no relevant supporting information exists in the retrieved documents.

This behavior is important for customer support systems because it reduces hallucination risk. Instead of creating unsupported answers, the assistant can indicate that the information is unavailable and direct users to contact support for additional assistance.

### 8.2 Strengths and Weaknesses


**List 3 strengths of the RAG system:**

1. The assistant generates answers using information retrieved from the Learnexity knowledge base instead of relying only on the language model's internal knowledge. This improves accuracy and reduces the possibility of fabricated responses.

2. Each response includes the source document used to answer the question. This improves trust because users and administrators can verify where the information originated.

3.The use of embeddings and ChromaDB allows users to ask questions naturally without needing to match exact keywords. For example, users can ask similar questions using different words and the system can still identify the relevant support documentation.

**List 3 weaknesses or limitations:**

1. The current question-answering model extracts information from retrieved text but does not perform advanced reasoning or generate highly conversational responses.

2. Dependence on Knowledge Base Quality
The assistant can only answer questions covered by the documents provided. Missing or outdated information may lead to incomplete responses.

3. Retrieval Challenges with Complex Questions
Questions requiring information from multiple documents may not always retrieve enough context. Improving retrieval strategies would be necessary for a production-level system.

### 8.3 Improvement Recommendations

**If I was to deploy this as a production system, what improvements would I make?**

Consider:
- Knowledge base improvements
The knowledge base should continue expanding with additional documents, including:

Detailed program curriculum pages
Instructor profiles
Upcoming cohort schedules
Student success stories
Career support resources
More detailed payment scenarios
Common customer conversations

The knowledge base should also have a regular review process to ensure information remains accurate and updated.




- Chunking strategy changes
Future improvements could include:

Splitting documents based on headings and sections
Keeping related paragraphs together
Creating smaller chunks for FAQ answers
Creating larger chunks for policy documents where context is important

This would improve retrieval accuracy by preserving meaningful information relationships.


- Retrieval enhancements
Retrieval Enhancements

The retrieval system could be improved by:

Increasing the number of retrieved documents for complex questions
Implementing hybrid search combining keyword search and semantic search
Adding reranking models to select the most relevant retrieved chunks
Filtering results by document category

For example, a question about refunds could prioritize policy documents before general company information.


- Generation improvements
The current system uses a question-answering extraction model. A production version would use a larger generative language model capable of:

Producing natural conversational responses
Handling multi-step questions
Summarizing information from multiple sources
Maintaining conversation history
Providing personalized responses


- User experience features
User Experience Improvements

Additional features could include:

Website chatbot integration
Conversation memory
User authentication
Multilingual support
Escalation to human support agents
Analytics dashboard showing common customer questions
Feedback buttons allowing users to rate responses


---

## Part 9: Demo Showcase

Create a polished demo of the assistant in action.

In [ ]:
# ============================================================
# Create a demo showcasing your assistant
# ============================================================

def run_demo(assistant: RAGAssistant, demo_questions: List[str]):
    """
    Run a polished demo of the Learnexity RAG customer support assistant.
    """

    print("\n" + "="*70)
    print(f"{PROJECT_INFO['topic']} - Knowledge Assistant Demo")
    print(f"   {PROJECT_INFO['description']}")
    print("="*70)


    for i, question in enumerate(demo_questions, 1):

        print(f"\n{'─'*70}")

        print(f"User Question {i}:")
        print(f"   \"{question}\"")

        print()


        # Run RAG pipeline
        result = assistant.ask(question)


        print("Assistant Response:")

        print(
            f"   {result['answer']}"
        )


        print()


        print(
            f"   Sources: {', '.join(result['sources'][:2])}"
        )


        print(
            f"   Confidence: {result['confidence']:.1%}"
        )


        print(
            f"   Confident Answer: {result['is_confident']}"
        )



    print(f"\n{'='*70}")

    print("Demo complete!")

    print("="*70)



# ============================================================
# Select 5 demonstration questions
# ============================================================

demo_questions = [

    # Demonstrates pricing retrieval
    "How much does the Artificial Intelligence Engineering Bootcamp cost?",


    # Demonstrates enrollment workflow retrieval
    "How do I enroll in a Learnexity program and what happens after I apply?",


    # Demonstrates technical troubleshooting retrieval
    "What should I do if I cannot log into my Learnexity account?",


    # Demonstrates policy retrieval
    "What is Learnexity's refund and cancellation policy?",


    # Demonstrates program information retrieval
    "What technology programs and services does Learnexity offer?"

]



# Run final demonstration

run_demo(
    assistant,
    demo_questions
)

---

## Submission Checklist

Before submitting, verify you have completed all requirements:

### Knowledge Base
- [ ] At least 8 documents in your knowledge base
- [ ] Each document is at least 150 words
- [ ] Documents cover diverse subtopics
- [ ] Content includes specific, factual information

### Technical Implementation
- [ ] Chunking function implemented with configurable parameters
- [ ] ChromaDB collection created and populated
- [ ] RAG pipeline class with all required methods
- [ ] Confidence-based response handling implemented
- [ ] Source attribution included in responses

### Testing
- [ ] At least 15 test questions
- [ ] Questions organized by category/difficulty
- [ ] At least 3 edge case questions included
- [ ] All test results documented

### Documentation
- [ ] Project declaration completed (Part 2)
- [ ] Chunking strategy justified (Part 4)
- [ ] Performance analysis completed (Part 8.1)
- [ ] Strengths and weaknesses identified (Part 8.2)
- [ ] Improvement recommendations provided (Part 8.3)
- [ ] Lessons learned documented (Part 8.4)

### Demo
- [ ] Demo showcase with 5 well-chosen questions
- [ ] All code cells executed with visible output



## Congratulations!

You have built a complete RAG-powered knowledge assistant from scratch!

### What You Demonstrated:

1. Creating and organizing a knowledge base
2. Implementing document chunking strategies
3. Using vector databases for semantic search
4. Building end-to-end RAG pipelines
5. Handling edge cases and uncertainty
6. Testing and analyzing system performance
7. Documenting technical decisions and learnings
